# OUTIL DE COTATION LC — Bibliothèque du musée d'Orsay

Usage courant, une notice à la fois ou en lot (CSV).


## 0 · Setup, bibliothèques, clé API et fonctions de base

In [ ]:
# Bibliothèques absentes de Colab par défaut :
#   ipywidgets   : boutons et champs de formulaire
#   scikit-learn : similarité cosinus entre vecteurs (embeddings)
#   isbnlib      : nettoyage et validation des ISBN
!pip install ipywidgets scikit-learn isbnlib -q

import os, re, json, time, requests
import pandas as pd
import numpy as np
import isbnlib
import ipywidgets as widgets
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity

# --- Clé API Mistral -------------------------------------------------------
# Lue dans les Secrets de Colab (icône clé, à gauche), sinon dans les variables d'environnement.
try:
    from google.colab import userdata
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except Exception:
    MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "")
if not MISTRAL_API_KEY:
    raise RuntimeError("MISTRAL_API_KEY absente : l'ajouter dans les Secrets Colab et activer "
                       "l'accès au notebook (icône clé, à gauche).")

# --- Réglages du modèle ----------------------------------------------------
MODELE_MISTRAL = "open-mistral-nemo"
TEMPERATURE = 0.1                            # proche de 0 : réponses stables
FICHIER_REPRISE = "embeddings_reprise.npy"   # sauvegarde intermédiaire des embeddings (§4)


def appeler_llm(prompt, tentatives=3):
    """Envoie un prompt à Mistral et renvoie le texte de la réponse.
    En cas d'erreur 429 (trop de requêtes), attend 10, 20 puis 30 s et réessaie."""
    for essai in range(tentatives):
        r = requests.post("https://api.mistral.ai/v1/chat/completions",
            headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
            json={"model": MODELE_MISTRAL, "messages": [{"role": "user", "content": prompt}],
                  "temperature": TEMPERATURE},
            timeout=30)
        if r.status_code != 429:
            break
        time.sleep(10 * (essai + 1))
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


def embeddings_mistral(textes, pause=1.0, tentatives=5, reprise=None):
    """Transforme une liste de textes en vecteurs (embeddings), par lots de 32.
    Avec reprise=<fichier>, l'avancement est sauvegardé en cours de route et un
    nouvel appel repart du dernier lot enregistré."""
    vecteurs = []
    if reprise and os.path.exists(reprise):
        vecteurs = list(np.load(reprise))

    for i in range(len(vecteurs), len(textes), 32):   # reprise après coupure (les coupures sont fréquentes)
        lot = textes[i:i + 32]
        corps = None
        for essai in range(tentatives):
            try:
                r = requests.post("https://api.mistral.ai/v1/embeddings",
                    headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
                    json={"model": "mistral-embed", "input": lot}, timeout=120)
                corps = r.json()
            except Exception as e:                    # coupure réseau, délai dépassé
                corps = {"message": str(e)}
            if "data" in corps:
                break
            attente = 15 * (essai + 1)                # les limites Mistral se libèrent en quelques dizaines de secondes
            print(f"  Lot {i}-{i+len(lot)} : échec {essai+1}/{tentatives} ")
            time.sleep(attente)
        else:
            # toutes les tentatives ont échoué
            if reprise:
                np.save(reprise, np.array(vecteurs))
                print(f"Avancement sauvegardé ({len(vecteurs)} vecteurs). "
                      f"Relancer la cellule.")
            raise RuntimeError(f"Échec définitif sur le lot {i}-{i+len(lot)} : {corps}")

        # tri sur "index" : garantit que les vecteurs suivent l'ordre des textes envoyés
        vecteurs.extend(d["embedding"] for d in sorted(corps["data"], key=lambda d: d["index"]))

        if reprise and (i // 32) % 50 == 0:           # sauvegarde tous les 50 lots
            np.save(reprise, np.array(vecteurs))
        if (i + 32) % 640 == 0:
            print(f"  Embeddings : {i+32}/{len(textes)}...")
        time.sleep(pause)

    return np.array(vecteurs, dtype=np.float32)       # float32 : fichier deux fois plus léger (problème de poids fréquent)


def fix_isbn(isbn):
    """Renvoie l'ISBN sous forme canonique (sans tirets, clé de contrôle vérifiée),
    ou None s'il est invalide. Validation via isbnlib, dans l'esprit du code de Belvadi 2023."""
    if not isbn or str(isbn).strip() == "":
        return None
    return isbnlib.get_canonical_isbn(str(isbn)) or None


# Cache de la recherche par ISBN : évite de réinterroger LC ou Open Library
# pour un ISBN déjà traité dans la session.
_cache_isbn = {}

print("Setup terminé.")

Setup terminé.


## 1 · Données (plan réduit, plan complet, tables, Clément)

In [ ]:
# Fichiers à déposer dans la session Colab avant de lancer cette cellule (voir le README) :
#   plan_lcc_orsay.csv                   plan réduit BM'O
#   plan_lcc_officiel_reel_updated.csv   barème LC complet, extrait des PDF de la Library of Congress
#   table_cutter_cinema.csv, table_cutter_litterature.csv   tables de cutters BM'O (amenées à évoluer)
#   embeddings_bareme_complet.npy        cache des embeddings (recalculé en §4 s'il manque)

INCLURE_SUGGESTIONS = False   # True : garde aussi les entrées "suggestion" des plans BM'O (non validées)

# keep_default_na=False : sinon pandas lit "NA" comme une valeur manquante, alors que
# c'est une classe LC (Architecture). Même réglage pour tous les CSV de l'outil.
df_plan_complet = pd.read_csv("plan_lcc_officiel_reel_updated.csv", sep=";", encoding="utf-8-sig",
                              keep_default_na=False, na_values=[""])
df_plan_lcc = pd.read_csv("plan_lcc_orsay.csv", sep=";", encoding="utf-8-sig",
                          dtype=str, keep_default_na=False)

if not INCLURE_SUGGESTIONS:
    df_plan_lcc = df_plan_lcc[df_plan_lcc["statut"] != "suggestion"].reset_index(drop=True)

assert (df_plan_lcc[["classe", "groupe"]] != "").all().all(), \
    "Plan réduit : des lignes n'ont pas de classe ou de groupe (vérifier le CSV)."


# --- Correspondance lettre Clément -> groupe du plan ----------------------
# La cote BM'O (ex. "4 O 12") contient une lettre qui donne la grande classe LC.
# La valeur est un groupe du plan réduit, pas toujours une classe unique :
# "P" regroupe PN, PQ et PR.
CORRESPONDANCES_CLEMENT = {
    "P": "ND",    # Peinture
    "S": "NB",    # Sculpture
    "E": "N",     # Expositions
    "F": "TR",    # Photographie
    "UV": "AM",   # Musées (+ VO, VLOU, VLUX, VP, VF, W : musées spécifiques, même logique)
    "C": "PN",    # Cinéma, confirmé par le lot déjà recoté à la BM'O
    "D": "NC",    # Dessin, arts graphiques (NC et non ND, qui est la peinture)
    "O": "NK",    # Objets d'art
    "A": "NA",    # Architecture
    "R": "N",     # Généralités (art)
    "T": "NX",    # Marché de l'art, collections, collectionneurs
    "L": "P",     # Littérature : groupe P = PN + PQ + PR
}

GROUPES_MULTICLASSES = {"P": ["PN", "PQ", "PR"]}

def classes_du_groupe(groupe):
    """Liste des classes LC d'un groupe (une seule, sauf pour la littérature)."""
    return GROUPES_MULTICLASSES.get(groupe, [groupe])

def libelle_groupe(groupe):
    """Texte affiché dans les prompts et l'interface : 'NK', ou 'P (PN, PQ, PR)'."""
    classes = classes_du_groupe(groupe)
    return groupe if classes == [groupe] else f"{groupe} ({', '.join(classes)})"


# --- Lettres qui ne mènent pas à une classe : traitement manuel -----------
# Disciplines réelles, mais absentes des classes BM'O actuelles.
CLASSES_HORS_PERIMETRE = {
    "G": "Gastronomie, arts de la table",
    "H": "Histoire",
    "I": "Imprimerie, édition",
    "J": "Journalisme, presse",
    "M": "Musique, théâtre, spectacles",
}

# Lettres de forme (catalogue commercial, périodique, usuel) et non de sujet :
# la lettre seule ne permet pas de déduire la classe LC.
CLASSES_FORME_SEULE = {
    "Q": "Catalogues commerciaux, manuels techniques",
    "X": "Périodiques, catalogues de ventes",
    "Z": "Usuels (dictionnaires, répertoires, etc.)",
}


# --- Lecture de la cote BM'O ----------------------------------------------
def extraire_lettre_clement(cote):
    """Lettre(s) de la cote BM'O : "4 O 12" -> "O", "12 UV 3" -> "UV"."""
    m = re.match(r'^\d+\s+([A-Z]{1,3})\s?', str(cote).strip())
    return m.group(1) if m else None

def determiner_classe(notice):
    """Groupe du plan pour cette notice, ou None si la lettre est absente ou inconnue.
    Contrairement à regle_niveau_0, ne gère pas les cas de traitement manuel."""
    lettre = extraire_lettre_clement(notice.get("Cote", ""))
    return CORRESPONDANCES_CLEMENT.get(lettre) if lettre else None

def est_monographie(notice):
    """Les monographies d'artiste (lettre B) suivent une branche à part (§2)."""
    return extraire_lettre_clement(notice.get("Cote", "")) == "B"

def regle_niveau_0(notice):
    """Niveau 0 : la lettre Clément donne la grande classe (confiance 1.0), jamais une
    cote complète. Renvoie None si la lettre est absente ou inconnue."""
    lettre = extraire_lettre_clement(notice.get("Cote", ""))
    if not lettre:
        return None
    if lettre in CORRESPONDANCES_CLEMENT:
        return {"cote_lc": CORRESPONDANCES_CLEMENT[lettre],
                "source": f"Niveau 0 (Clément {lettre})", "confiance": 1.0}
    if lettre in CLASSES_HORS_PERIMETRE:
        return {"cote_lc": None, "source": "Traitement manuel (hors périmètre)",
                "confiance": 0.0,
                "note": f"Clément {lettre} ({CLASSES_HORS_PERIMETRE[lettre]}) : "
                        f"discipline hors des classes BM'O actuelles, à arbitrer avec Matthieu."}
    if lettre in CLASSES_FORME_SEULE:
        return {"cote_lc": None, "source": "Traitement manuel (catégorie de forme)",
                "confiance": 0.0,
                "note": f"Clément {lettre} ({CLASSES_FORME_SEULE[lettre]}) : "
                        f"catégorie de forme, ne détermine pas la grande classe à elle seule."}
    return None

print(f"Plan réduit : {len(df_plan_lcc)} entrées | Plan complet : {len(df_plan_complet)} entrées")

Plan réduit : 584 entrées | Plan complet : 74858 entrées


## 2 · Branche monographie (Cote Clément B)

In [ ]:
# Cote cible d'une monographie d'artiste :
#     NY [identifiant artiste] [type d'ouvrage] [auteur] [année]
# Cette branche produit NY + artiste + type ; le cutter auteur et l'année sont
# ajoutés à l'affichage (_cote_finale, §9).
# L'identifiant artiste n'est pas encore défini par la BM'O (état au 22/09) :
# le placeholder [ARTISTE] est à remplacer à la main.

def detecter_type_ouvrage(notice):
    """Devine, d'après le titre et le sujet, laquelle des 4 catégories de Matthieu
    s'applique. Renvoie (nom du type, numéro utilisé dans la cote) :
        1 = catalogue d'exposition      2 = catalogue raisonné ou de collection
        3 = écrits de l'artiste         4 = biographie ou étude critique (par défaut)
    Les tests sont appliqués dans l'ordre : le premier qui correspond l'emporte."""
    titre = str(notice.get("Titre", "")).lower()
    auteur = str(notice.get("Auteur", "")).strip()
    sujet = str(notice.get("Sujet", "")).strip().lower()

    if "exposition" in titre:
        return "catalogue_exposition", "1"
    if "catalogue raisonn" in titre:                 # avec ou sans accent
        return "catalogue_raisonne", "2"
    if "collection" in titre or "catalogue" in titre:
        return "catalogue_collection", "2"

    # Écrits de l'artiste : l'auteur figure aussi dans les sujets.
    # "Monet, Claude" -> "monet"
    nom_auteur = auteur.split(",")[0].split("(")[0].strip().lower()
    if len(nom_auteur) > 2 and nom_auteur in sujet:  # > 2 : évite les faux positifs sur les noms très courts
        return "ecrits_artiste", "3"

    return "biographie_critique", "4"


def traiter_monographie(notice):
    """Cote partielle pour une monographie d'artiste, à compléter par la BM'O."""
    type_ouvrage, cutter_type = detecter_type_ouvrage(notice)
    return {"cote_lc": f"NY [ARTISTE] .{cutter_type}",   # type noté en cutter numérique (.1 à .4)
            "type_ouvrage": type_ouvrage,
            "source": "Branche monographie (partielle)", "confiance": 0.5,
            "note": f"Type d'ouvrage détecté : {type_ouvrage.replace('_', ' ')} (à vérifier). "
                    f"Identifiant artiste à définir (en attente BM'O)."}

## 3 · Niveau 1 — ISBN (LC SRU puis Open Library)

In [ ]:
# Si l'ISBN est valide, on cherche la cote réelle au catalogue de la Library of
# Congress, puis dans Open Library. C'est le niveau le plus fiable : la cote vient
# d'une notice existante et elle est déjà complète (cutter et année inclus),
# l'interface n'y ajoute donc rien (§9).

# Serveur SRU de la LC (ancienne adresse, toujours documentée : http://lx2.loc.gov:210/lcdb)
LOC_SRU_BASE = "https://lx2.loc.gov/sru/lcdb"

# Open Library demande aux applications de s'identifier (nom + contact),
# sinon les requêtes peuvent être limitées.
EMAIL_CONTACT = "adresse@exemple.fr"   # à remplacer par une adresse réelle
HEADERS_OPENLIBRARY = {"User-Agent": f"CotationBMO/1.0 ({EMAIL_CONTACT})"}


def isbn_vers_lcc_locsru(isbn):
    """Cherche l'ISBN au catalogue de la LC ; renvoie la cote LCC ou None.
    Toute erreur (réseau, délai, réponse inattendue) vaut "pas de résultat" :
    la cascade passe alors à Open Library."""
    try:
        r = requests.get(LOC_SRU_BASE, params={
            "version": "1.1", "operation": "searchRetrieve",
            "query": f"bath.isbn={isbn}",
            "maximumRecords": 1,
            "recordSchema": "mods"}, timeout=10)
        r.raise_for_status()

        # <classification authority="lcc">...</classification>, avec ou sans préfixe "mods:"
        m = re.search(r'<[^>]*classification[^>]*lcc[^>]*>([^<]+)<', r.text)
        return m.group(1).strip() if m else None
    except Exception:
        return None


def isbn_vers_lcc_openlibrary(isbn):
    """Repli : cherche l'ISBN dans Open Library ; renvoie la cote LCC ou None."""
    try:
        r = requests.get("https://openlibrary.org/api/books",
                         params={"bibkeys": f"ISBN:{isbn}", "format": "json", "jscmd": "data"},
                         headers=HEADERS_OPENLIBRARY, timeout=10)
        data = r.json().get(f"ISBN:{isbn}", {})
        cotes = data.get("classifications", {}).get("lc_classifications", [])
        return cotes[0] if cotes else None
    except Exception:
        return None


def regle_niveau_1(notice):
    """Niveau 1 : renvoie {"cote_lc", "source", "confiance"} si l'ISBN donne une cote,
    sinon None."""
    isbn = fix_isbn(notice.get("ISBN", ""))
    if not isbn:
        return None
    if isbn in _cache_isbn:
        return _cache_isbn[isbn]

    cote = isbn_vers_lcc_locsru(isbn)
    source = "Niveau 1 (LC SRU)"
    if not cote:
        cote = isbn_vers_lcc_openlibrary(isbn)
        source = "Niveau 1 (Open Library)"
    if not cote:
        return None                               # l'échec n'est pas mis en cache : il peut être passager

    _cache_isbn[isbn] = {"cote_lc": cote, "source": source, "confiance": 0.9}
    return _cache_isbn[isbn]

# Test rapide des deux sources
print(isbn_vers_lcc_locsru("9780521779241"))
print(isbn_vers_lcc_openlibrary("9780521779241"))

B1545.Z7 C36 2006
B1545.Z7 .C36 2005


## 4 · Embeddings du barème complet (calculés une fois)

In [ ]:
# Un embedding transforme un texte en vecteur : deux textes de sens proche donnent
# deux vecteurs proches (similarité cosinus proche de 1). On calcule un vecteur par
# ligne du barème LC, puis on cherche les lignes les plus proches d'une notice.
# Le calcul prend plus d'une heure : il est fait une fois, puis sauvegardé dans deux
# fichiers qui vont ensemble :
#   embeddings_bareme_complet.npy        les vecteurs
#   embeddings_bareme_complet_index.csv  les entrées du barème, dans le même ordre
# Si le barème (plan_lcc_officiel_reel_updated.csv) change, supprimer ces deux fichiers.

FICHIER_EMB = "embeddings_bareme_complet.npy"
FICHIER_EMB_INDEX = "embeddings_bareme_complet_index.csv"

# Seules les classes atteignables depuis une lettre Clément sont vectorisées.
CLASSES_EMBEDDING = sorted({c for g in CORRESPONDANCES_CLEMENT.values() for c in classes_du_groupe(g)})


def _verifier_cache(idx, emb):
    """Vérifie que le cache correspond au barème actuel, sinon lève une erreur."""
    if len(idx) != len(emb):
        raise ValueError(f"cache incohérent : {len(idx)} lignes d'index pour {len(emb)} vecteurs")
    attendues = set(CLASSES_EMBEDDING) & set(df_plan_complet["classe"])
    manquantes = attendues - set(idx["classe"])
    if manquantes:                 # typiquement : cache calculé avant l'ajout de nouvelles classes
        raise ValueError(f"cache périmé : classes absentes {sorted(manquantes)}")


def preparer_embeddings_bareme():
    """Renvoie (index, vecteurs). Recharge les fichiers s'ils existent, sinon calcule et sauvegarde."""
    if os.path.exists(FICHIER_EMB) and os.path.exists(FICHIER_EMB_INDEX):
        emb = np.load(FICHIER_EMB)
        idx = pd.read_csv(FICHIER_EMB_INDEX, sep=";", encoding="utf-8-sig",
                          keep_default_na=False, na_values=[""])
        _verifier_cache(idx, emb)
        print(f"Embeddings rechargés depuis le disque ({len(idx)} entrées).")
        return idx, emb

    # reset_index : la ligne n de l'index doit correspondre au vecteur n
    bareme = df_plan_complet[df_plan_complet["classe"].isin(CLASSES_EMBEDDING)].reset_index(drop=True)
    print(f"Calcul des embeddings pour {len(bareme)} entrées ({', '.join(CLASSES_EMBEDDING)})...")
    textes = (bareme["cote_range"].astype(str) + " " + bareme["description"].astype(str)).tolist()

    emb = embeddings_mistral(textes, reprise=FICHIER_REPRISE)

    np.save(FICHIER_EMB, emb)
    idx = bareme[["classe", "cote_range", "description"]]
    idx.to_csv(FICHIER_EMB_INDEX, sep=";", index=False, encoding="utf-8-sig")
    if os.path.exists(FICHIER_REPRISE):
        os.remove(FICHIER_REPRISE)
    print(f"Embeddings calculés et sauvegardés ({len(idx)} entrées).")
    return idx, emb


def bareme_disponible(classe):
    """Vrai si le barème complet contient des entrées pour cette classe ou ce groupe."""
    return index_bareme["classe"].isin(classes_du_groupe(classe)).any()


def trois_candidats_embedding(titre, sujets, classe, exclure=None):
    """Source 1 : les 3 entrées du barème les plus proches de la notice.
    Pour les 3 suivantes, rappeler la fonction avec exclure = cotes déjà montrées."""
    sous = index_bareme[index_bareme["classe"].isin(classes_du_groupe(classe))]
    if sous.empty:
        return []
    emb_notice = embeddings_mistral([f"{titre} {sujets}"])[0]
    # sous.index : position des vecteurs correspondants dans le tableau complet
    scores = cosine_similarity(emb_notice.reshape(1, -1), embeddings_bareme[sous.index])[0]
    resultats = []
    for i in np.argsort(scores)[::-1]:
        cote = sous.iloc[i]["cote_range"]
        if exclure and cote in exclure:
            continue
        resultats.append({"cote": cote, "description": sous.iloc[i]["description"]})
        if len(resultats) == 3:
            break
    return resultats


def trois_candidats_llm_direct(titre, sujets, classe, orientation="", eviter=None):
    """Source 2 : le modèle propose 3 sous-classes d'après sa connaissance générale du
    LCC, sans le barème (trop volumineux pour le prompt).
    orientation : précision libre tapée par le bibliothécaire (supplée un sujet RAMEAU absent).
    eviter      : cotes déjà proposées, à ne pas répéter (bouton "autre proposition")."""
    ligne_orientation = f"\nPrécision donnée par le bibliothécaire : {orientation}\n" if orientation else ""
    ligne_eviter = f"\nCes cotes ont déjà été proposées, n'en propose pas d'autres identiques : {', '.join(eviter)}\n" if eviter else ""

    prompt = f"""Tu es un bibliothécaire spécialisé en histoire de l'art, avec une
bonne connaissance du système Library of Congress Classification (LCC).

La grande classe de ce document est déjà déterminée : {libelle_groupe(classe)}.

Document :
Titre : {titre}
Sujets RAMEAU : {sujets}
{ligne_orientation}{ligne_eviter}
D'après ta connaissance générale du LCC (sans référentiel fourni), propose
3 sous-classes plausibles dans {libelle_groupe(classe)} pour ce document, classées de la
plus à la moins probable, avec une justification très courte pour chacune.
Donne uniquement la sous-classe (lettres et numéro, par exemple PN1993), sans cutter ni année.

Réponds UNIQUEMENT au format JSON suivant :
{{"choix": [{{"cote": "...", "justification": "..."}}, {{"cote": "...", "justification": "..."}}, {{"cote": "...", "justification": "..."}}]}}"""

    texte_brut = appeler_llm(prompt)
    # retire les balises ```json éventuelles autour de la réponse
    texte = re.sub(r'^```(?:json)?\s*|\s*```$', '', texte_brut.strip(), flags=re.MULTILINE).strip()
    try:
        choix = json.loads(texte).get("choix") or []
    except (json.JSONDecodeError, AttributeError):
        return []
    return [{"cote": ch.get("cote"), "description": ch.get("justification", "")}
            for ch in choix[:3] if isinstance(ch, dict) and ch.get("cote")]


# Chargement en fin de cellule : si le calcul échoue (limite de débit, coupure), les
# fonctions ci-dessus restent définies et l'outil fonctionne sans la colonne
# "par similarité".
try:
    index_bareme, embeddings_bareme = preparer_embeddings_bareme()
except Exception as e:
    print(f"Embeddings indisponibles pour cette session : {e}")
    print("Si le message parle d'un cache, supprimer les deux fichiers embeddings_bareme_complet*")
    print("puis relancer. Sinon, relancer la cellule : le calcul reprend là où il s'est arrêté.")
    index_bareme = pd.DataFrame(columns=["classe", "cote_range", "description"])
    embeddings_bareme = np.zeros((0, 1))

Calcul des embeddings pour 48101 entrées (AM, N, NA, NB, NC, ND, NK, NX, PN, PQ, PR, TR)...
  Embeddings : 640/48101...
  Embeddings : 1280/48101...
  Embeddings : 1920/48101...
  Embeddings : 2560/48101...
  Embeddings : 3200/48101...
  Embeddings : 3840/48101...
  Embeddings : 4480/48101...
  Embeddings : 5120/48101...
  Embeddings : 5760/48101...
  Embeddings : 6400/48101...
  Embeddings : 7040/48101...
  Embeddings : 7680/48101...
  Embeddings : 8320/48101...
  Embeddings : 8960/48101...
  Embeddings : 9600/48101...
  Embeddings : 10240/48101...
  Embeddings : 10880/48101...
  Embeddings : 11520/48101...
  Embeddings : 12160/48101...
  Embeddings : 12800/48101...
  Embeddings : 13440/48101...
  Embeddings : 14080/48101...
  Embeddings : 14720/48101...
  Embeddings : 15360/48101...
  Embeddings : 16000/48101...
  Embeddings : 16640/48101...
  Embeddings : 17280/48101...
  Embeddings : 17920/48101...
  Embeddings : 18560/48101...
  Embeddings : 19200/48101...
  Embeddings : 19840/481

## 4bis · Pool few-shot (optionnel, décoché par défaut)

In [ ]:
# Few-shot : on montre au modèle quelques notices déjà cotées et proches de celle à
# coter. Le pool est un CSV de notices BM'O (colonnes Titre ; Sujets ; Cote_LC).
# La composition du pool compte plus que sa taille : un pool déséquilibré entre
# classes a dégradé les résultats sur NX.

FICHIER_POOL_FEWSHOT = "pool_fewshot_bmo.csv"

df_pool_fewshot, embeddings_fewshot = None, None     # None = option few-shot inactive

if os.path.exists(FICHIER_POOL_FEWSHOT):
    try:
        df_pool_fewshot = pd.read_csv(FICHIER_POOL_FEWSHOT, sep=";", encoding="utf-8-sig",
                                      dtype=str, keep_default_na=False)
        # Pool petit : pas de cache sur disque
        embeddings_fewshot = embeddings_mistral((df_pool_fewshot["Titre"] + " " + df_pool_fewshot["Sujets"]).tolist())
        print(f"Pool few-shot chargé : {len(df_pool_fewshot)} notices.")
    except Exception as e:
        print(f"Pool few-shot indisponible ({e}) : l'option few-shot restera inactive.")
        df_pool_fewshot, embeddings_fewshot = None, None
else:
    print(f"Pas de {FICHIER_POOL_FEWSHOT} trouvé : l'option few-shot restera inactive tant qu'il n'est pas fourni.")


def exemples_similaires(titre, sujets, k=3):
    """Renvoie les k notices du pool les plus proches de la notice à coter,
    sous la forme {"titre", "sujets", "cote_lc"}. Liste vide si le pool est indisponible."""
    if df_pool_fewshot is None or df_pool_fewshot.empty:
        return []
    emb = embeddings_mistral([f"{titre} {sujets}"])[0]
    scores = cosine_similarity(emb.reshape(1, -1), embeddings_fewshot)[0]
    plus_proches = np.argsort(scores)[::-1][:k]
    # titres et sujets tronqués pour garder un prompt court
    return [{"titre": df_pool_fewshot.iloc[i]["Titre"][:80],
             "sujets": df_pool_fewshot.iloc[i]["Sujets"][:100],
             "cote_lc": df_pool_fewshot.iloc[i]["Cote_LC"]} for i in plus_proches]

Pas de pool_fewshot_bmo.csv trouvé -- l'option few-shot restera inactive tant qu'il n'est pas fourni.


## 5 · LLM — plan réduit BM'O

In [ ]:
# La grande classe est connue (lettre Clément). Le modèle choisit une sous-classe
# dans le plan réduit BM'O, avec les consignes de la bibliothèque.

def retirer_balises_json(texte_brut):
    """Retire les balises ```json ... ``` dont les modèles entourent parfois leur réponse."""
    return re.sub(r'^```(?:json)?\s*|\s*```$', '', texte_brut.strip(), flags=re.MULTILINE).strip()


def lire_reponse_json(texte_brut):
    """Extrait la sous-classe de la réponse du modèle, attendue sous la forme
    {"sous_classe": "..."}."""
    texte = retirer_balises_json(texte_brut)
    try:                                       # réponse entièrement en JSON
        return json.loads(texte).get("sous_classe", "")
    except (json.JSONDecodeError, AttributeError):
        pass
    m = re.search(r'\{[^{}]*"sous_classe"[^{}]*\}', texte)     # JSON au milieu de texte
    if m:
        try:
            return json.loads(m.group(0)).get("sous_classe", "")
        except json.JSONDecodeError:
            pass
    return texte                               # à défaut, la réponse brute, lisible par la personne


def plan_reduit_pour_classe(classe):
    """Plan réduit d'un groupe en texte : une ligne par cote, avec les consignes BM'O.
    Filtre sur "groupe" et non sur "classe" : P réunit PN, PQ et PR."""
    sous = df_plan_lcc[df_plan_lcc["groupe"] == classe]
    if sous.empty:
        return None
    lignes = []
    for _, r in sous.iterrows():
        ligne = f"  {r['cote_range']:16} → {r['description']}"
        if r["commentaire"]:
            ligne += f"  [consigne BM'O : {r['commentaire']}]"
        lignes.append(ligne)
    return "\n".join(lignes)


def construire_prompt_reduit(notice, classe, avec_rameau, exemples_fewshot, eviter=None):
    """Assemble le prompt : rôle, plan de cotation, exemples éventuels, notice à coter."""
    titre = str(notice.get("Titre", ""))
    sujets = str(notice.get("Sujet", ""))
    plan = plan_reduit_pour_classe(classe) or "(pas de plan)"
    nom_classe = libelle_groupe(classe)

    ex_str = ""
    if exemples_fewshot:
        ex_str = "\nExemples de notices déjà cotées :\n"
        for i, e in enumerate(exemples_fewshot, 1):
            ex_str += f"\nEx {i} : {e['titre']}\n  Sujets : {e['sujets']}\n  Cote : {e['cote_lc']}\n"

    # Sans l'option RAMEAU, le modèle ne voit que le titre.
    consigne = ("en te basant EN PRIORITÉ sur les sujets RAMEAU." if avec_rameau
                else "en te basant sur le titre du document.")
    ligne_sujets = f"Sujets RAMEAU : {sujets}" if avec_rameau else "(sujets RAMEAU non communiqués)"

    # Relance ("autre proposition") : signaler la cote refusée
    ligne_eviter = ""
    if eviter:
        ligne_eviter = f"\nLa proposition {eviter} a déjà été jugée peu satisfaisante : propose une autre sous-classe, plausible mais différente.\n"

    return f"""Tu es un bibliothécaire spécialisé en histoire de l'art.

La grande classe de ce document est déjà déterminée : {nom_classe}.

Ton travail : choisir la SOUS-CLASSE la plus précise dans {nom_classe},
{consigne}

Plan de cotation pour {nom_classe} :
{plan}
{ex_str}{ligne_eviter}
Document :
Titre : {titre}
{ligne_sujets}

Réponds UNIQUEMENT au format JSON suivant, sans aucun texte avant ou après :
{{"sous_classe": "<la cote de sous-classe la plus adaptée>"}}"""


def regle_niveau_3_reduit(notice, classe, avec_rameau, avec_fewshot, eviter=None):
    """Niveau 3 : le modèle choisit dans le plan réduit. La cote renvoyée n'est pas
    vérifiée contre le plan : elle est à valider (confiance 0.7)."""
    exemples = exemples_similaires(notice.get("Titre", ""), notice.get("Sujet", ""), k=3) if avec_fewshot else []
    prompt = construire_prompt_reduit(notice, classe, avec_rameau, exemples, eviter=eviter)
    texte_brut = appeler_llm(prompt)
    cote = lire_reponse_json(texte_brut)
    return {"cote_lc": cote, "source": "Niveau 3 (LLM, plan réduit)", "confiance": 0.7}

## 6 · Cutter (tables BM'O, puis table de Cutter LC)

In [ ]:
# Le cutter range les ouvrages par nom d'auteur sous une même cote (ex. .C37).
# Ordre de recherche : table BM'O cinéma, table BM'O littérature, puis calcul.
#
# Le calcul suit la table de Cutter de la Library of Congress (Classification and
# Shelflisting Manual, G 63 : https://www.loc.gov/aba/pcc/053/table.html), code adapté
# de https://github.com/veralvx/csan. C'est une approximation : à la LC, le numéro
# final dépend des ouvrages déjà en rayon (d'où "approx." dans la source).

import unicodedata


def normaliser(t):
    """Majuscules sans accents : 'Éluard' -> 'ELUARD'. Les ligatures (Œ, Æ) sont
    développées à part, la décomposition Unicode ne les traite pas."""
    t = str(t).upper().replace("Œ", "OE").replace("Æ", "AE")
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t.strip()


def nom_de_base(nom):
    """Patronyme normalisé : 'Balzac, de' -> 'BALZAC'.
    Sans virgule ('Prénom Nom'), on prend le dernier mot, ce qui est faux pour
    certains noms composés."""
    n = normaliser(nom)
    if "," in n:
        return n.split(",")[0].strip().replace("'", "").replace("\u2019", "")
    mots = n.replace("'", "").replace("\u2019", "").split()
    return mots[-1] if mots else n


def lire_table_cutter(fichier, colonne_cutter):
    """Lit une table de cutters BM'O (colonnes : nom ; cutter). Table absente : table vide.
    keep_default_na=False : sinon des noms comme "Nan" ou "Null" seraient lus comme vides."""
    if os.path.exists(fichier):
        df = pd.read_csv(fichier, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    else:
        df = pd.DataFrame(columns=["nom", colonne_cutter])
    df["nom_base"] = df["nom"].apply(nom_de_base)     # clé de recherche
    return df


df_cutter_cinema = lire_table_cutter("table_cutter_cinema.csv", "cutter")
df_cutter_litt = lire_table_cutter("table_cutter_litterature.csv", "identifiant_lc")


# --- Table de Cutter de la LC, codée lettre par lettre --------------------
# Chaque chaîne donne un chiffre pour chaque symbole de VL, dans le même ordre.
VL = "ABCDEFGHIJKLMNOPQRSTUVWXYZ01234567890"
# TF : catégorie de la 1re lettre. 1 = voyelle, 2 = S, 3 = Q, 4 = autre consonne, 5 = chiffre
TF = "1444144414444414342414444455555555555"
# T2 : chiffre selon la 2e lettre, pour chaque catégorie (lignes 1, 2 et 4 de la table LC).
# X = pas de chiffre ; Q = noms en Q (2-29 pour Qa-Qt) ; U = le "u" de "Qu", ignoré.
T2 = {"1": "22233333333445566788999999XXXXXXXXXX",
      "2": "22334445555566666667899999XXXXXXXXXX",
      "3": "QQQQQQQQQQQQQQQQQQQQUUUUUUXXXXXXXXXX",
      "4": "33334444555555666777888899XXXXXXXXXX",
      "5": "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"}
# T3 : lettres suivantes, table d'expansion de la LC :
# a-d = 3, e-h = 4, i-l = 5, m-o = 6, p-s = 7, t-v = 8, w-z = 9.
T3 = "33334444555566677778889999"
CHIFFRES_VALIDES = "23456789Q"


def calculer_cutter_lc(nom, longueur=4):
    """Cutter approché d'un patronyme d'après la table de la LC : la première lettre,
    puis un chiffre par lettre suivante, jusqu'à `longueur` caractères. 'CAROU' -> 'C376'."""
    nom = nom.upper().strip()
    if len(nom) < 2:
        return nom
    cutter = nom[0]
    pos = VL.find(nom[0])
    categorie = TF[pos] if pos >= 0 else "5"
    pos = VL.find(nom[1])                                  # 2e lettre : chiffre selon la catégorie
    if pos >= 0 and T2[categorie][pos] in CHIFFRES_VALIDES:
        cutter += T2[categorie][pos]
    for lettre in nom[2:]:                                 # lettres suivantes : table d'expansion
        if len(cutter) >= longueur:
            break
        pos = VL.find(lettre)
        if 0 <= pos < len(T3):
            cutter += T3[pos]
    return cutter


def calculer_cutter(nom):
    """Renvoie {"cutter", "source"} pour un nom d'auteur : tables BM'O, puis calcul.
    En cas d'homonymes dans une table, la 1re ligne est retenue."""
    nb = nom_de_base(nom)
    m = df_cutter_cinema[df_cutter_cinema["nom_base"] == nb]
    if not m.empty:
        return {"cutter": m.iloc[0]["cutter"], "source": "Table BM'O cinéma"}
    m = df_cutter_litt[df_cutter_litt["nom_base"] == nb]
    if not m.empty:
        return {"cutter": m.iloc[0]["identifiant_lc"], "source": "Table BM'O littérature"}
    return {"cutter": "." + calculer_cutter_lc(nb), "source": "Table de Cutter LC (approx.)"}

## 7 · Options — à cocher avant de lancer

In [ ]:
# Les cases sont lues au moment du clic : pas besoin de relancer la cellule.

COURT = {"description_width": "initial"}
LARGE = widgets.Layout(width="auto")          # évite que les libellés longs soient coupés

# Plan réduit : cotes validées par la BM'O (plus sûr). LCC complet : toute la
# classification LC (plus fin, mais la personne tranche entre plusieurs propositions).
case_plan_complet = widgets.Checkbox(value=False, layout=LARGE, style=COURT,
    description="Chercher dans le LCC complet (plus fin ; sinon plan réduit BM'O)")

case_rameau = widgets.Checkbox(value=True, layout=LARGE, style=COURT,
    description="Prioriser les sujets RAMEAU sur le titre")

# Few-shot : grisé si le pool n'a pas pu être chargé (§4bis)
pool_ok = df_pool_fewshot is not None
case_fewshot = widgets.Checkbox(value=False, disabled=not pool_ok, layout=LARGE, style=COURT,
    description="Utiliser des exemples de notices déjà cotées (few-shot)"
                + ("" if pool_ok else " (indisponible : pool absent)"))

case_cutter = widgets.Checkbox(value=True, layout=LARGE, style=COURT,
    description="Calculer le cutter auteur")

display(widgets.VBox([widgets.HTML("<b>Options de cotation</b>"),
                      case_plan_complet, case_rameau, case_fewshot, case_cutter]))

## 8 · Cascade de cotation

In [ ]:
# Ordre de la cascade : monographie (§2) -> ISBN (§3) -> lettre Clément (§1)
# -> sous-classe par le modèle, en plan réduit (§5) ou en LCC complet (§4, §9).
# La première règle qui donne un résultat l'emporte.

def cascade_avant_llm(notice):
    """Monographie d'artiste (§2) ou cote trouvée par l'ISBN (§3) : résultat définitif,
    sans passer par le modèle. Renvoie None si aucune des deux ne s'applique."""
    if est_monographie(notice):
        return traiter_monographie(notice)
    r1 = regle_niveau_1(notice)
    if r1:
        return r1
    return None


def coter_notice(notice, plan_complet, avec_rameau, avec_fewshot, eviter=None):
    """Point d'entrée de la cascade, utilisé par §9 (une notice) et §10 (CSV).
    Renvoie soit un résultat définitif ({"cote_lc": ..., "source": ...}), soit
    {"candidats": True, "classe": ...} quand la personne doit choisir (LCC complet).

    eviter : cote jugée peu satisfaisante, pour obtenir une autre proposition en plan
    réduit. Sans effet en LCC complet, qui a son propre bouton "voir d'autres"."""

    # Sur une relance, inutile de revérifier monographie et ISBN : ils n'avaient rien donné.
    if not eviter:
        resultat = cascade_avant_llm(notice)
        if resultat:
            return resultat

    # La lettre Clément oriente la suite mais ne donne jamais une cote complète.
    # Un cas de traitement manuel, en revanche, est un résultat définitif.
    r0 = regle_niveau_0(notice)
    if r0 and r0.get("cote_lc"):
        classe = r0["cote_lc"]
    elif r0:
        return r0
    else:
        classe = determiner_classe(notice)

    if not classe:
        return {"cote_lc": None, "source": "Classe indéterminée", "confiance": 0.0,
                "note": "Cote Clément absente ou non reconnue : classement impossible sans grande classe."}

    if plan_complet:
        # les deux sources sont calculées côté interface (afficher_deux_sources, §9)
        return {"candidats": True, "classe": classe}
    else:
        return regle_niveau_3_reduit(notice, classe, avec_rameau, avec_fewshot, eviter=eviter)

## 9 · Mode « une seule notice »

In [ ]:
import html

champ_cote = widgets.Text(description="Cote (BM'O) :", placeholder="4 P 72")
champ_titre = widgets.Text(description="Titre :")
champ_nom = widgets.Text(description="Nom :", placeholder="Carou")
champ_prenom = widgets.Text(description="Prénom :", placeholder="Alain")
champ_sujet = widgets.Textarea(layout=widgets.Layout(width="90%", height="80px"))
champ_isbn = widgets.Text(description="ISBN :")
champ_annee = widgets.Text(description="Année :")
bouton_coter = widgets.Button(description="Coter cette notice", button_style="primary")

# Sous Colab, clear_output + display dans un callback duplique ou masque les sorties :
# on remplace directement la liste .children des VBox.
zone_resultat = widgets.VBox()


def _html(txt):
    """Insère du HTML déjà construit."""
    return widgets.HTML(f"<div style='margin:2px 0'>{txt}</div>")

def _texte(txt):
    """Affiche du texte brut, échappé (< > &) pour qu'un titre ne casse pas l'affichage."""
    return _html(html.escape(str(txt)))

def _label(txt):
    return widgets.HTML(f"<div style='margin-top:6px'><b>{txt}</b></div>")

def _erreur(e):
    return _html(f"<span style='color:#b00'>Erreur : {html.escape(str(e))}</span>")

def _remplir(boite, construire, attente="Calcul en cours..."):
    """Affiche un message d'attente, puis remplace le contenu de la boîte
    par la liste de widgets renvoyée par construire()."""
    boite.children = [_html(f"<i>{attente}</i>")]
    try:
        boite.children = construire()
    except Exception as e:
        boite.children = [_erreur(e)]


def _cote_finale(cote_choisie, notice):
    """Ajoute le cutter auteur et l'année à une cote de sous-classe."""
    cutter_str = ""
    if case_cutter.value and notice.get("Auteur"):
        cutter_str = f" {calculer_cutter(notice['Auteur'])['cutter']}"
    annee = str(notice.get("Annee", "")).strip()
    return f"{cote_choisie}{cutter_str}" + (f" {annee}" if annee and annee != "nan" else "")


def afficher_choix(boite_choix, cote_choisie, notice, source):
    # Niveau 1 : cote déjà complète, on n'ajoute ni cutter ni année (sinon ils seraient en double).
    cote = cote_choisie if source.startswith("Niveau 1") else _cote_finale(cote_choisie, notice)
    boite_choix.children = [
        _html(f"Cote proposée : <b>{html.escape(cote)}</b>"),
        _texte(f"Source : {source}")]


def _widgets_candidats(candidats, notice, source_label, boite_choix):
    """Un bouton 'Choisir' par candidat, précédé de sa description."""
    elements = []
    for cand in candidats:
        btn = widgets.Button(description=f"Choisir {cand['cote']}", layout=widgets.Layout(width="90%"))
        btn.on_click(lambda b, c=cand["cote"]: afficher_choix(boite_choix, c, notice, source_label))
        elements += [_texte(cand["description"]), btn]
    return elements


def bloc_llm(classe, notice, boites, orientation="", deja_proposees=None):
    """Colonne de gauche du LCC complet : propositions du modèle, sans le barème."""
    deja_proposees = deja_proposees or []
    titre = _html("--- Propositions du modèle (connaissance générale du LCC) ---")
    candidats = trois_candidats_llm_direct(notice.get("Titre", ""), notice.get("Sujet", ""), classe,
                                           orientation=orientation, eviter=deja_proposees)
    if not candidats:
        return [titre, _html("(le modèle n'a pas produit de proposition exploitable)")]
    deja = deja_proposees + [c["cote"] for c in candidats]
    btn = widgets.Button(description="Autre proposition (modèle)", button_style="warning",
                         layout=widgets.Layout(width="90%"))
    btn.on_click(lambda b: _remplir(boites["llm"],
                                    lambda: bloc_llm(classe, notice, boites, orientation, deja)))
    return [titre] + _widgets_candidats(candidats, notice, "LCC complet (LLM, choix humain)",
                                        boites["choix"]) + [btn]


def bloc_embedding(classe, notice, boites, orientation="", rang=0, deja_proposees=None):
    """Colonne de droite du LCC complet : cotes les plus proches par similarité."""
    deja_proposees = deja_proposees or []
    if not bareme_disponible(classe):
        return [_html("--- Par similarité avec le barème ---"),
                _texte(f"Barème complet non extrait pour {libelle_groupe(classe)} : "
                      f"pas de proposition par similarité.")]
    titre = _html("--- Par similarité avec le barème (navigable) ---" if rang == 0
                  else f"--- Par similarité, suivants (rang {rang+1}-{rang+3}) ---")
    sujets_enrichis = f"{notice.get('Sujet', '')} {orientation}".strip()
    candidats = trois_candidats_embedding(notice.get("Titre", ""), sujets_enrichis, classe,
                                          exclure=deja_proposees)
    if not candidats:
        return [titre, _html("Plus aucun candidat disponible.")]
    deja = deja_proposees + [c["cote"] for c in candidats]
    btn = widgets.Button(description="Voir d'autres (embedding)", button_style="warning",
                         layout=widgets.Layout(width="90%"))
    btn.on_click(lambda b: _remplir(boites["emb"],
                                    lambda: bloc_embedding(classe, notice, boites, orientation, rang + 3, deja)))
    return [titre] + _widgets_candidats(candidats, notice, "LCC complet (embedding, choix humain)",
                                        boites["choix"]) + [btn]


def afficher_deux_sources(classe, notice):
    """Vue LCC complet : les deux colonnes (modèle / similarité) et le champ de précision."""
    boites = {"choix": widgets.VBox(), "llm": widgets.VBox(), "emb": widgets.VBox()}
    champ_orientation = widgets.Textarea(
        placeholder="Optionnel, ex. : \"porte sur le lien entre cinéma et littérature, pas le cinéma en général\"",
        layout=widgets.Layout(width="90%", height="60px"))
    btn_relancer = widgets.Button(description="Relancer les deux sources avec cette précision",
                                  button_style="info", layout=widgets.Layout(width="90%"))

    def lancer(orientation=""):
        boites["choix"].children = []
        _remplir(boites["llm"], lambda: bloc_llm(classe, notice, boites, orientation),
                 "Interrogation du modèle...")
        _remplir(boites["emb"], lambda: bloc_embedding(classe, notice, boites, orientation),
                 "Calcul des similarités...")

    btn_relancer.on_click(lambda b: lancer(champ_orientation.value))
    zone_resultat.children = [boites["choix"], _label("Préciser :"), champ_orientation, btn_relancer,
                              boites["llm"], boites["emb"]]
    lancer()


def afficher_resultat(resultat, notice, deja_rejetees=None):
    """Résultat d'une cotation en plan réduit (Niveau 0/1/3, monographie, ou manuel)."""
    deja_rejetees = deja_rejetees or []
    boite_choix = widgets.VBox()
    afficher_choix(boite_choix, resultat.get("cote_lc") or "(non trouvée)", notice, resultat.get("source", ""))
    elements = [boite_choix]
    if resultat.get("note"):
        elements.append(_texte(f"Note : {resultat['note']}"))

    # Seul le Niveau 3 propose une relance : les autres résultats sont définitifs.
    if resultat.get("source", "").startswith("Niveau 3") and not case_plan_complet.value:
        rejetees = deja_rejetees + [resultat.get("cote_lc")]
        classe = determiner_classe(notice)

        def au_clic_autre_reduit(b):
            zone_resultat.children = [_html("<i>Nouvelle proposition en cours...</i>")]
            try:
                nouveau = coter_notice(notice, False, case_rameau.value, case_fewshot.value,
                                       eviter=", ".join(c for c in rejetees if c))
            except Exception as e:
                nouveau = {"cote_lc": None, "source": f"Erreur : {e}"}
            afficher_resultat(nouveau, notice, rejetees)

        btn_autre = widgets.Button(description="Autre proposition (toujours le plan réduit)",
                                   button_style="warning", layout=widgets.Layout(width="90%"))
        btn_autre.on_click(au_clic_autre_reduit)
        btn_complet = widgets.Button(description="Voir le LCC complet",
                                     button_style="warning", layout=widgets.Layout(width="90%"))
        btn_complet.on_click(lambda b: afficher_deux_sources(classe, notice))
        elements += [btn_autre, btn_complet]

    zone_resultat.children = elements


def au_clic_coter(b):
    """Bouton 'Coter cette notice' : construit la notice à partir du formulaire,
    lance la cascade (§8) et affiche le résultat."""
    auteur = champ_nom.value.strip()
    if champ_prenom.value.strip():
        auteur += f", {champ_prenom.value.strip()}"
    notice = {"Cote": champ_cote.value, "Titre": champ_titre.value, "Auteur": auteur,
              "Sujet": champ_sujet.value, "ISBN": champ_isbn.value, "Annee": champ_annee.value}
    zone_resultat.children = [_html("<i>Cotation en cours...</i>")]
    try:
        resultat = coter_notice(notice, case_plan_complet.value, case_rameau.value, case_fewshot.value)
    except Exception as e:
        zone_resultat.children = [_erreur(e)]
        return
    if "candidats" in resultat:
        afficher_deux_sources(resultat["classe"], notice)
    else:
        afficher_resultat(resultat, notice)


bouton_coter.on_click(au_clic_coter)

display(widgets.VBox([
    champ_cote,
    champ_titre,
    champ_nom,
    champ_prenom,
    _label("Sujets RAMEAU :"),
    champ_sujet,
    champ_isbn,
    champ_annee,
    bouton_coter,
    zone_resultat
]))

## 10 · Mode « fichier CSV »

In [ ]:
champ_fichier_csv = widgets.Text(description="Fichier CSV :", placeholder="notices_a_coter.csv")
bouton_importer = widgets.Button(description="Importer un fichier", icon="upload")
bouton_traiter_csv = widgets.Button(description="Traiter le fichier", button_style="primary")
zone_resultat_csv = widgets.Output()


def au_clic_importer(b):
    """Ouvre le sélecteur de fichiers de Colab et remplit le champ avec le nom déposé.
    files.upload() est appelé hors du bloc "with zone_resultat_csv" : à l'intérieur,
    la fenêtre de sélection ne s'affichait pas toujours."""
    from google.colab import files
    depose = files.upload()
    zone_resultat_csv.clear_output()
    with zone_resultat_csv:
        if depose:
            champ_fichier_csv.value = list(depose)[0]
            print(f"Fichier déposé : {champ_fichier_csv.value}")
        else:
            print("Aucun fichier reçu.")

bouton_importer.on_click(au_clic_importer)


def au_clic_traiter_csv(b):
    """Cote chaque ligne du CSV avec la même cascade que le formulaire (§8) et écrit un
    fichier de sortie avec des colonnes en plus : cote_lc / source / a_trancher, ou
    candidat_1..6 quand plusieurs propositions restent à départager."""
    zone_resultat_csv.clear_output()
    # dtype=str, keep_default_na=False : sinon une case vide devient NaN ("nan" dans
    # les prompts) et une année comme "2004" devient 2004.0.
    df_notices = pd.read_csv(champ_fichier_csv.value, sep=";", encoding="utf-8-sig",
                             dtype=str, keep_default_na=False)
    lignes = []
    for i, row in df_notices.iterrows():
        notice = row.to_dict()
        ligne = dict(notice)      # les colonnes d'origine sont conservées
        try:
            resultat = coter_notice(notice, case_plan_complet.value, case_rameau.value, case_fewshot.value)

            if case_cutter.value and notice.get("Auteur"):
                c = calculer_cutter(notice["Auteur"])
                ligne["cutter"] = c["cutter"]
                ligne["cutter_source"] = c["source"]
                # pour une cote de Niveau 1, ce cutter est indicatif : la cote LC est déjà complète

            if "candidats" in resultat:
                # LCC complet : pas de choix automatique, les candidats des deux sources
                # sont écrits dans le fichier pour être départagés
                classe = resultat["classe"]
                candidats = trois_candidats_llm_direct(notice.get("Titre", ""), notice.get("Sujet", ""), classe)
                if bareme_disponible(classe):
                    candidats += trois_candidats_embedding(notice.get("Titre", ""), notice.get("Sujet", ""), classe)
                for j, cand in enumerate(candidats, 1):
                    ligne[f"candidat_{j}"] = cand["cote"]
                    ligne[f"candidat_{j}_description"] = cand["description"]
                ligne["a_trancher"] = "oui"
            else:
                ligne["cote_lc"] = resultat.get("cote_lc")
                ligne["source"] = resultat.get("source")
                ligne["a_trancher"] = "non"
        except Exception as e:
            # une erreur sur une notice n'interrompt pas le lot : elle est notée dans "source"
            ligne["source"] = f"Erreur : {e}"
            ligne["a_trancher"] = "oui"

        lignes.append(ligne)
        if (i + 1) % 20 == 0:
            with zone_resultat_csv:
                print(f"  {i+1}/{len(df_notices)} notices traitées...")

    df_sortie = pd.DataFrame(lignes)
    fichier_sortie = "cote_" + os.path.basename(champ_fichier_csv.value)
    df_sortie.to_csv(fichier_sortie, sep=";", index=False, encoding="utf-8-sig")
    with zone_resultat_csv:
        print(f"\nTerminé. Fichier écrit : {fichier_sortie}")
        print(f"{(df_sortie['a_trancher']=='oui').sum()} notice(s) à trancher manuellement (colonnes candidat_*).")

bouton_traiter_csv.on_click(au_clic_traiter_csv)

display(champ_fichier_csv, bouton_importer, bouton_traiter_csv, zone_resultat_csv)

Text(value='', description='Fichier CSV :', placeholder='notices_a_coter.csv')

Button(description='Importer un fichier', icon='upload', style=ButtonStyle())

Button(button_style='primary', description='Traiter le fichier', style=ButtonStyle())

Output()